# Model evaluation and re-training with AdaPT on Cifar10 dataset

In this notebook you can evaluate different approximate multipliers on various models based on Cifar10 dataset

Steps:
* Select models to load 
* Select number of threads to use
* Choose approximate multiplier 
* Load model for evaluation
* Load dataset
* Run model calibration for quantization
* Run model evaluation
* Run approximate-aware re-training
* Rerun model evaluation

**Note**:
* This notebook should be run on a X86 machine

* Please make sure you have run the installation steps first

In [1]:
import os
import zipfile
import torch

import requests
from torch.utils.data import DataLoader
from torchvision import transforms as T
from torchvision.datasets import FashionMNIST
from tqdm import tqdm
import torch.nn as nn

## Select models to load 

The weights must be downloaded in state_dicts folder.


In [2]:
from models.resnet import resnet18, resnet34, resnet50
from models.vgg import vgg11_bn, vgg13_bn, vgg19_bn
from models.densenet import densenet121, densenet161, densenet169
from models.inception import inception_v3 # slow, propably bad cifar10 implementation of inception for PT

## Select number of threads to use

For optimal performance set them as the number of your cpu threads (not cpu cores)

In [3]:
threads = 40
torch.set_num_threads(threads)

#maybe better performance
%env OMP_PLACES=cores
%env OMP_PROC_BIND=close
%env OMP_WAIT_POLICY=active

env: OMP_PLACES=cores
env: OMP_PROC_BIND=close
env: OMP_WAIT_POLICY=active


## Choose approximate multiplier 

Two approximate multipliers are already provided

**mul8s_acc** - (header file: mul8s_acc.h)   <--  default

**mul8s_1L2H** - (header file: mul8s_1L2H.h)



In order to use your custom multiplier you need to use the provided tool (LUT_generator) to easily create the C++ header for your multiplier. Then you just place it inside the adapt/cpu-kernels/axx_mults folder. The name of the axx_mult here must match the name of the header file. The same axx_mult is used in all layers. 

Tip: If you want explicitly to set for each layer a different axx_mult you must do it from the model definition using the respective AdaPT_Conv2d class of each layer.

In [4]:
axx_mult = 'mul8s_acc'

## Load model for evaluation

Jit compilation method loads 'on the fly' the C++ extentions of the approximate multipliers. Then the pytorch model is loaded

In [5]:
model = resnet50(pretrained=False, axx_mult = axx_mult)

model.eval() # for evaluation

Using /root/.cache/torch_extensions as PyTorch extensions root...
Emitting ninja build file /root/.cache/torch_extensions/PyInit_conv2d_mul8s_acc/build.ninja...
Building extension module PyInit_conv2d_mul8s_acc...
Allowing ninja to set a default number of workers... (overridable by setting the environment variable MAX_JOBS=N)
ninja: no work to do.
Loading extension module PyInit_conv2d_mul8s_acc...
Using /root/.cache/torch_extensions as PyTorch extensions root...
No modifications detected for re-loaded extension module PyInit_conv2d_mul8s_acc, skipping build step...
Loading extension module PyInit_conv2d_mul8s_acc...
Using /root/.cache/torch_extensions as PyTorch extensions root...
No modifications detected for re-loaded extension module PyInit_conv2d_mul8s_acc, skipping build step...
Loading extension module PyInit_conv2d_mul8s_acc...
Using /root/.cache/torch_extensions as PyTorch extensions root...
No modifications detected for re-loaded extension module PyInit_conv2d_mul8s_acc, skip

ResNet(
  (conv1): AdaPT_Conv2d(
    3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False
    (quantizer): TensorQuantizer(8bit per-tensor amax=dynamic calibrator=HistogramCalibrator quant)
    (quantizer_w): TensorQuantizer(8bit per-tensor amax=dynamic calibrator=HistogramCalibrator quant)
  )
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): AdaPT_Conv2d(
        64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False
        (quantizer): TensorQuantizer(8bit per-tensor amax=dynamic calibrator=HistogramCalibrator quant)
        (quantizer_w): TensorQuantizer(8bit per-tensor amax=dynamic calibrator=HistogramCalibrator quant)
      )
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): AdaPT_Conv2d(
      

## Load dataset


In [6]:
def val_dataloader(mean = (0.2860), std = (0.3530)):

    transform = T.Compose(
        [
            T.ToTensor(),
            T.Normalize(mean, std),
        ]
    )
    
    dataset = FashionMNIST(root="datasets/fashionmnist_data", train=False, download=True, transform=transform)
    dataloader = DataLoader(
        dataset,
        batch_size=128,
        num_workers=0,
        drop_last=True,
        pin_memory=False,
    )
    return dataloader

transform = T.Compose(
        [
            T.RandomCrop(28, padding=4),
            T.RandomHorizontalFlip(),
            T.ToTensor(),
            T.Normalize(mean = (0.2860), std = (0.3530)),
        ]
    )
dataset = FashionMNIST(root="datasets/fashionmnist_data", train=True, download=True, transform=transform)

evens = list(range(0, len(dataset), 10))
trainset_1 = torch.utils.data.Subset(dataset, evens)

data = val_dataloader()

# data_t is used for calibration purposes and is a subset of train-set
data_t = DataLoader(trainset_1, batch_size=128,
                                            shuffle=False, num_workers=0)


In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.models import resnet50
import time

standard_model = resnet50(pretrained=False)

# KORREKTUR 1: Das exakte 3x3-Raster für die "Augen" nutzen
standard_model.conv1 = nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1, bias=False)

# KORREKTUR 2: Den Ausgang des Modells strikt auf 10 Klassen begrenzen
standard_model.fc = nn.Linear(standard_model.fc.in_features, 10)

# Modell auf die RTX 4060 schieben
standard_model = standard_model.to("cuda")

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(standard_model.parameters(), lr=0.01, momentum=0.9)

# Den Trainloader für das Dataset wieder definieren
trainloader = torch.utils.data.DataLoader(dataset, batch_size=128, shuffle=True, num_workers=0)

epochen = 5
print("🚀 Starte KORRIGIERTES Vorab-Training auf der RTX 4060...")

for epoch in range(epochen):
    standard_model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    start_time = time.time()
    
    for i, (inputs, labels) in enumerate(trainloader):
        inputs, labels = inputs.to("cuda"), labels.to("cuda")
        
        optimizer.zero_grad()
        outputs = standard_model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
    epoch_time = time.time() - start_time
    epoch_acc = 100. * correct / total
    print(f"Epoche {epoch+1}/{epochen} | Loss: {running_loss/len(trainloader):.4f} | Train Acc: {epoch_acc:.2f}% | Zeit: {epoch_time:.1f}s")

# Das perfekt angepasste Gehirn sofort speichern!
torch.save(standard_model.state_dict(), "fashionmnist_resnet50_klug.pt")
print("✅ Training beendet und neues, perfekt passendes Gehirn gespeichert!")

🚀 Starte KORRIGIERTES Vorab-Training auf der RTX 4060...
Epoche 1/5 | Loss: 0.9675 | Train Acc: 66.76% | Zeit: 45.3s
Epoche 2/5 | Loss: 0.4659 | Train Acc: 82.80% | Zeit: 45.1s
Epoche 3/5 | Loss: 0.3895 | Train Acc: 85.61% | Zeit: 45.2s
Epoche 4/5 | Loss: 0.3488 | Train Acc: 87.09% | Zeit: 45.2s
Epoche 5/5 | Loss: 0.3198 | Train Acc: 88.22% | Zeit: 45.2s
✅ Training beendet und neues, perfekt passendes Gehirn gespeichert!


In [8]:
# Speichert die trainierten Gewichte als Datei ab
torch.save(standard_model.state_dict(), "fashionmnist_resnet50_klug.pt")
print("💾 Gewichte erfolgreich gespeichert!")

💾 Gewichte erfolgreich gespeichert!


In [7]:
import torch

model = resnet50(pretrained=False, axx_mult=axx_mult, in_channel=1)

# 1. Wir nutzen dein originales AdaPT-Modell, das weiter oben schon definiert wurde
model = model.to("cpu")

# 2. Pflanze das gespeicherte, kluge Gehirn ein
kluges_gehirn = torch.load("fashionmnist_resnet50_klug.pt")
model.load_state_dict(kluges_gehirn, strict=False) 

print("🧠 Gehirn-Transplantation erfolgreich!")

Using /root/.cache/torch_extensions as PyTorch extensions root...
No modifications detected for re-loaded extension module PyInit_conv2d_mul8s_acc, skipping build step...
Loading extension module PyInit_conv2d_mul8s_acc...
Using /root/.cache/torch_extensions as PyTorch extensions root...
No modifications detected for re-loaded extension module PyInit_conv2d_mul8s_acc, skipping build step...
Loading extension module PyInit_conv2d_mul8s_acc...
Using /root/.cache/torch_extensions as PyTorch extensions root...
No modifications detected for re-loaded extension module PyInit_conv2d_mul8s_acc, skipping build step...
Loading extension module PyInit_conv2d_mul8s_acc...
Using /root/.cache/torch_extensions as PyTorch extensions root...
No modifications detected for re-loaded extension module PyInit_conv2d_mul8s_acc, skipping build step...
Loading extension module PyInit_conv2d_mul8s_acc...
Using /root/.cache/torch_extensions as PyTorch extensions root...
No modifications detected for re-loaded ex

🧠 Gehirn-Transplantation erfolgreich!


## Run model calibration for quantization

Calibrates the quantization parameters 

Need to re-run it each time the model changes

In [8]:
from pytorch_quantization import nn as quant_nn
from pytorch_quantization import calib

def collect_stats(model, data_loader, num_batches):
     """Feed data to the network and collect statistic"""

     # Enable calibrators
     for name, module in model.named_modules():
         if isinstance(module, quant_nn.TensorQuantizer):
             if module._calibrator is not None:
                 module.disable_quant()
                 module.enable_calib()
             else:
                 module.disable()

     for i, (image, _) in tqdm(enumerate(data_loader), total=num_batches):
         model(image.cpu())
         if i >= num_batches:
             break

     # Disable calibrators
     for name, module in model.named_modules():
         if isinstance(module, quant_nn.TensorQuantizer):
             if module._calibrator is not None:
                 module.enable_quant()
                 module.disable_calib()
             else:
                 module.enable()

def compute_amax(model, **kwargs):
 # Load calib result
 for name, module in model.named_modules():
     if isinstance(module, quant_nn.TensorQuantizer):
         if module._calibrator is not None:
             if isinstance(module._calibrator, calib.MaxCalibrator):
                 module.load_calib_amax()
             else:
                 module.load_calib_amax(**kwargs)
         print(F"{name:40}: {module}")
 model.cpu()

# It is a bit slow since we collect histograms on CPU
with torch.no_grad():
    stats = collect_stats(model, data_t, num_batches=2)
    amax = compute_amax(model, method="percentile", percentile=99.99)
    
    # optional - test different calibration methods
    #amax = compute_amax(model, method="mse")
    #amax = compute_amax(model, method="entropy")
    

100%|██████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:13<00:00,  6.73s/it]
W0604 20:09:25.421417 129398487697216 tensor_quantizer.py:173] Disable HistogramCalibrator
W0604 20:09:25.421752 129398487697216 tensor_quantizer.py:173] Disable HistogramCalibrator
W0604 20:09:25.421987 129398487697216 tensor_quantizer.py:173] Disable HistogramCalibrator
W0604 20:09:25.422159 129398487697216 tensor_quantizer.py:173] Disable HistogramCalibrator
W0604 20:09:25.422377 129398487697216 tensor_quantizer.py:173] Disable HistogramCalibrator
W0604 20:09:25.422552 129398487697216 tensor_quantizer.py:173] Disable HistogramCalibrator
W0604 20:09:25.422709 129398487697216 tensor_quantizer.py:173] Disable HistogramCalibrator
W0604 20:09:25.422851 129398487697216 tensor_quantizer.py:173] Disable HistogramCalibrator
W0604 20:09:25.423001 129398487697216 tensor_quantizer.py:173] Disable HistogramCalibrator
W0604 20:09:25.423160 129398487697216 tensor_qu

W0604 20:09:25.438270 129398487697216 tensor_quantizer.py:173] Disable HistogramCalibrator
W0604 20:09:25.438404 129398487697216 tensor_quantizer.py:173] Disable HistogramCalibrator
W0604 20:09:25.438838 129398487697216 tensor_quantizer.py:173] Disable HistogramCalibrator
W0604 20:09:25.438960 129398487697216 tensor_quantizer.py:173] Disable HistogramCalibrator
W0604 20:09:25.439103 129398487697216 tensor_quantizer.py:173] Disable HistogramCalibrator
W0604 20:09:25.439220 129398487697216 tensor_quantizer.py:173] Disable HistogramCalibrator
W0604 20:09:25.439335 129398487697216 tensor_quantizer.py:173] Disable HistogramCalibrator
W0604 20:09:25.439474 129398487697216 tensor_quantizer.py:173] Disable HistogramCalibrator
W0604 20:09:25.439596 129398487697216 tensor_quantizer.py:173] Disable HistogramCalibrator
W0604 20:09:25.439721 129398487697216 tensor_quantizer.py:173] Disable HistogramCalibrator
W0604 20:09:25.439848 129398487697216 tensor_quantizer.py:173] Disable HistogramCalibrator

W0604 20:09:25.457595 129398487697216 tensor_quantizer.py:237] Load calibrated amax, shape=torch.Size([]).
W0604 20:09:25.457816 129398487697216 tensor_quantizer.py:237] Load calibrated amax, shape=torch.Size([]).
W0604 20:09:25.458038 129398487697216 tensor_quantizer.py:237] Load calibrated amax, shape=torch.Size([]).
W0604 20:09:25.458276 129398487697216 tensor_quantizer.py:237] Load calibrated amax, shape=torch.Size([]).
W0604 20:09:25.458667 129398487697216 tensor_quantizer.py:237] Load calibrated amax, shape=torch.Size([]).
W0604 20:09:25.458953 129398487697216 tensor_quantizer.py:237] Load calibrated amax, shape=torch.Size([]).
W0604 20:09:25.459225 129398487697216 tensor_quantizer.py:237] Load calibrated amax, shape=torch.Size([]).
W0604 20:09:25.459500 129398487697216 tensor_quantizer.py:237] Load calibrated amax, shape=torch.Size([]).
W0604 20:09:25.459734 129398487697216 tensor_quantizer.py:237] Load calibrated amax, shape=torch.Size([]).
W0604 20:09:25.460173 129398487697216

conv1.quantizer                         : TensorQuantizer(8bit per-tensor amax=2.0217 calibrator=HistogramCalibrator quant)
conv1.quantizer_w                       : TensorQuantizer(8bit per-tensor amax=0.7546 calibrator=HistogramCalibrator quant)
layer1.0.conv1.quantizer                : TensorQuantizer(8bit per-tensor amax=6.1603 calibrator=HistogramCalibrator quant)
layer1.0.conv1.quantizer_w              : TensorQuantizer(8bit per-tensor amax=0.6475 calibrator=HistogramCalibrator quant)
layer1.0.conv2.quantizer                : TensorQuantizer(8bit per-tensor amax=4.3033 calibrator=HistogramCalibrator quant)
layer1.0.conv2.quantizer_w              : TensorQuantizer(8bit per-tensor amax=0.2695 calibrator=HistogramCalibrator quant)
layer1.0.conv3.quantizer                : TensorQuantizer(8bit per-tensor amax=4.9337 calibrator=HistogramCalibrator quant)
layer1.0.conv3.quantizer_w              : TensorQuantizer(8bit per-tensor amax=0.3292 calibrator=HistogramCalibrator quant)
layer1.0

## Run model evaluation

Tip: observe how the execution becomes faster and faster with each batch as the CPU achieves better cache re-use on the LUT table

In [9]:
import timeit
correct = 0
total = 0

model.eval()
start_time = timeit.default_timer()
with torch.no_grad():
    for iteraction, (images, labels) in tqdm(enumerate(data), total=len(data)):
        images, labels = images.to("cpu"), labels.to("cpu")
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
print(timeit.default_timer() - start_time)
print('Accuracy of the network on the 10000 test images: %.4f %%' % (
    100 * correct / total))

100%|████████████████████████████████████████████████████████████████████████████████████████████████| 78/78 [04:16<00:00,  3.29s/it]

256.7028705669982
Accuracy of the network on the 10000 test images: 89.3830 %


## Run approximate-aware re-training


In [10]:
from adapt.references.classification.train import evaluate, train_one_epoch, load_data

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.0001)
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.1)

# finetune the model for one epoch based on data_t subset 
train_one_epoch(model, criterion, optimizer, data_t, "cpu", 0, 1)



Epoch: [0]  [ 0/47]  eta: 0:14:30  lr: 0.0001  img/s: 6.914492535105315  loss: 0.3418 (0.3418)  acc1: 85.9375 (85.9375)  acc5: 100.0000 (100.0000)  time: 18.5199  data: 0.0081  max mem: 90
Epoch: [0]  [ 1/47]  eta: 0:14:07  lr: 0.0001  img/s: 6.985576559626499  loss: 0.2550 (0.2984)  acc1: 85.9375 (87.5000)  acc5: 100.0000 (100.0000)  time: 18.4254  data: 0.0077  max mem: 90
Epoch: [0]  [ 2/47]  eta: 0:13:46  lr: 0.0001  img/s: 7.007740781671842  loss: 0.2588 (0.2852)  acc1: 89.0625 (88.5417)  acc5: 100.0000 (100.0000)  time: 18.3745  data: 0.0075  max mem: 90
Epoch: [0]  [ 3/47]  eta: 0:13:26  lr: 0.0001  img/s: 7.04307191861929  loss: 0.2588 (0.2911)  acc1: 89.0625 (89.0625)  acc5: 100.0000 (100.0000)  time: 18.3262  data: 0.0075  max mem: 90
Epoch: [0]  [ 4/47]  eta: 0:13:07  lr: 0.0001  img/s: 7.0279686679873175  loss: 0.3087 (0.2977)  acc1: 89.0625 (88.9062)  acc5: 100.0000 (100.0000)  time: 18.3050  data: 0.0075  max mem: 90
Epoch: [0]  [ 5/47]  eta: 0:12:48  lr: 0.0001  img/s: 7

Epoch: [0]  [44/47]  eta: 0:00:54  lr: 0.0001  img/s: 7.007394761768888  loss: 0.2866 (0.3050)  acc1: 89.0625 (88.6111)  acc5: 100.0000 (99.8611)  time: 18.2794  data: 0.0073  max mem: 90
Epoch: [0]  [45/47]  eta: 0:00:36  lr: 0.0001  img/s: 7.031099640139851  loss: 0.2866 (0.3052)  acc1: 89.0625 (88.6039)  acc5: 100.0000 (99.8641)  time: 18.2752  data: 0.0073  max mem: 90
Epoch: [0]  [46/47]  eta: 0:00:18  lr: 0.0001  img/s: 6.79437421440933  loss: 0.2837 (0.3038)  acc1: 89.0625 (88.6333)  acc5: 100.0000 (99.8667)  time: 18.1905  data: 0.0073  max mem: 90
Epoch: [0] Total time: 0:14:17


## Rerun model evaluation

In [11]:
correct = 0
total = 0

model.eval()
start_time = timeit.default_timer()
with torch.no_grad():
    for iteraction, (images, labels) in tqdm(enumerate(data), total=len(data)):
        images, labels = images.to("cpu"), labels.to("cpu")
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
print(timeit.default_timer() - start_time)
print('Accuracy of the network on the 10000 test images: %.4f %%' % (
    100 * correct / total))

100%|████████████████████████████████████████████████████████████████████████████████████████████████| 78/78 [04:15<00:00,  3.28s/it]

255.46200887800296
Accuracy of the network on the 10000 test images: 89.0625 %
